# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anjds22/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
!pip -q install duckdb huggingface_hub

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
""")

print("DuckDB connection ready!")

DuckDB connection ready!


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions and reason codes

The action queue prioritises content for human review using observable search-performance signals. Items with low current impressions and/or poor average search position receive higher priority. These signals identify candidates for investigation, not guaranteed improvement opportunities.

Each recommendation includes a priority score, a reason code, and a suggested review action. `LOW_VISIBILITY` indicates low impressions, `WEAK_POSITION` indicates a poor average position, and `LOW_VISIBILITY_AND_WEAK_POSITION` indicates both signals are present. The queue is intended to help reviewers decide where to investigate first; it does not automatically prescribe content changes.

In [8]:
from pathlib import Path

print("Files in outputs:")
for f in Path("/content/work/outputs").glob("*"):
    print(f.name, f.stat().st_size, "bytes")

Files in outputs:


In [9]:
import pandas as pd
from pathlib import Path

monthly = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        AVG(gsc_avg_position) AS avg_position
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/'
        'fact_content_daily_performance/**/*.parquet'
    )
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
      AND gsc_data_available = TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

monthly["low_visibility"] = monthly["impressions"] < 100
monthly["weak_position"] = monthly["avg_position"] >= 20

monthly["reason_code"] = "OTHER_REVIEW"
monthly.loc[monthly["low_visibility"], "reason_code"] = "LOW_VISIBILITY"
monthly.loc[monthly["weak_position"], "reason_code"] = "WEAK_POSITION"
monthly.loc[
    monthly["low_visibility"] & monthly["weak_position"],
    "reason_code"
] = "LOW_VISIBILITY_AND_WEAK_POSITION"

monthly["priority_score"] = (
    monthly["low_visibility"].astype(int)
    + monthly["weak_position"].astype(int)
)

monthly["suggested_action"] = "REVIEW_PERFORMANCE_AND_SEARCH_INTENT"

queue = monthly.sort_values(
    ["priority_score", "impressions", "avg_position"],
    ascending=[False, True, False],
    na_position="last"
).reset_index(drop=True)

queue.insert(0, "rank", range(1, len(queue) + 1))

print("Total content items:", len(queue))
display(queue.head(20))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total content items: 176738


,rank,client_hash_id,content_hash_id,impressions,clicks,avg_position,low_visibility,weak_position,reason_code,priority_score,suggested_action
0,1,client_08a6a72ff48e62c0,content_9e8c3b83214c180d,1.0,0.0,309.0,True,True,LOW_VISIBILITY_AND_WEAK_POSITION,2,REVIEW_PERFORMANCE_AND_SEARCH_INTENT
1,2,client_3ffa76342f366962,content_06589faf15cc8488,1.0,0.0,297.0,True,True,LOW_VISIBILITY_AND_WEAK_POSITION,2,REVIEW_PERFORMANCE_AND_SEARCH_INTENT
2,3,client_3ffa76342f366962,content_36cc2bda86ee726a,1.0,0.0,292.0,True,True,LOW_VISIBILITY_AND_WEAK_POSITION,2,REVIEW_PERFORMANCE_AND_SEARCH_INTENT
3,4,client_08a6a72ff48e62c0,content_11187e07e5ee9f43,1.0,0.0,286.0,True,True,LOW_VISIBILITY_AND_WEAK_POSITION,2,REVIEW_PERFORMANCE_AND_SEARCH_INTENT
4,5,client_3ffa76342f366962,content_efce4eda2b012964,1.0,0.0,283.0,True,True,LOW_VISIBILITY_AND_WEAK_POSITION,2,REVIEW_PERFORMANCE_AND_SEARCH_INTENT
5,6,client_3ffa76342f366962,content_0cec599cfeab8b7f,1.0,0.0,280.0,True,True,LOW_VISIBILITY_AND_WEAK_POSITION,2,REVIEW_PERFORMANCE_AND_SEARCH_INTENT
6,7,client_23a62021009f63c4,content_61b375eafb1d4c27,1.0,0.0,271.0,True,True,LOW_VISIBILITY_AND_WEAK_POSITION,2,REVIEW_PERFORMANCE_AND_SEARCH_INTENT
7,8,client_f623b01661d4bfe4,content_fc468c5940d16ea3,1.0,0.0,262.0,True,True,LOW_VISIBILITY_AND_WEAK_POSITION,2,REVIEW_PERFORMANCE_AND_SEARCH_INTENT
8,9,client_3ffa76342f366962,content_3758dd311e8033f7,1.0,0.0,260.0,True,True,LOW_VISIBILITY_AND_WEAK_POSITION,2,REVIEW_PERFORMANCE_AND_SEARCH_INTENT
9,10,client_3ffa76342f366962,content_d1b44ca865290810,1.0,0.0,258.0,True,True,LOW_VISIBILITY_AND_WEAK_POSITION,2,REVIEW_PERFORMANCE_AND_SEARCH_INTENT


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use and limits

This playbook is intended for analysts or content teams who need a prioritised list of content items to investigate. It uses historical search-performance measurements to identify low-visibility items and items with a poor average position.

The recommendations are decision-support signals, not instructions to automatically rewrite, remove, or publish content. Performance can be affected by seasonality, search demand, data coverage, and other factors not captured by the model.

The model was evaluated on a sampled, time-aware holdout. Its measured performance should not be assumed to generalise to every client, content type, or future period. The analysis does not establish that acting on a recommendation will cause impressions or clicks to increase.

In [10]:
# Summarise the queue by reason code and priority score.

reason_summary = (
    queue.groupby("reason_code")
    .size()
    .reset_index(name="items")
)

priority_summary = (
    queue.groupby("priority_score")
    .size()
    .reset_index(name="items")
)

print("Total content items:", len(queue))

print("\nReason-code summary:")
display(reason_summary)

print("\nPriority-score summary:")
display(priority_summary)

Total content items: 176738

Reason-code summary:


,reason_code,items
0,LOW_VISIBILITY,54338
1,LOW_VISIBILITY_AND_WEAK_POSITION,20959
2,OTHER_REVIEW,77369
3,WEAK_POSITION,24072



Priority-score summary:


,priority_score,items
0,0,77369
1,1,78410
2,2,20959


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review and no-go list

Before acting on a recommendation, a human reviewer should verify that the content is still relevant, the data is sufficiently complete, the search intent is understood, and the proposed change is appropriate for the page.

Reviewers should check for seasonal demand, recent publication or migration changes, unusual impressions, and potentially misleading averages. Recommendations should be checked against the actual content and its purpose.

Do not automatically delete pages, rewrite or publish content, change important URLs, or make client-facing claims based only on this score. Do not infer private client details, expose client identities or URLs, or claim that the model proves a search-engine ranking mechanism. All changes require human approval.

In [11]:
# Check the queue for fields that should not be used as public identifiers.
print("Columns in the queue:")
print(queue.columns.tolist())

sensitive_name_checks = [
    col for col in queue.columns
    if any(term in col.lower() for term in ["url", "domain", "query", "email", "name"])
]
print("Potentially sensitive column-name checks:", sensitive_name_checks)

assert "client_hash_id" in queue.columns
assert "content_hash_id" in queue.columns
print("Identifiers are hashed IDs; do not publish the queue's raw rows.")

Columns in the queue:
['rank', 'client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'avg_position', 'low_visibility', 'weak_position', 'reason_code', 'priority_score', 'suggested_action']
Potentially sensitive column-name checks: []
Identifiers are hashed IDs; do not publish the queue's raw rows.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring and retraining triggers

The queue should be reviewed regularly and rebuilt when new performance data becomes available. Monitor changes in impression and position distributions, missing data, the proportion of items receiving each reason code, and model performance on later time periods.

Re-evaluate the model if precision or F1 declines materially against the baseline, if data coverage changes, or if the content population or measurement process changes. A new model should only replace the existing approach after evaluation on a suitable later-period holdout.

Retraining should not happen automatically just because a calendar date has passed. Changes should be documented, validated, and reviewed by a person before the new recommendations are used.

In [12]:
# Record a simple monitoring snapshot of the current queue.
monitoring_snapshot = pd.DataFrame([{
    "snapshot_month": "2026-03",
    "queue_rows": len(queue),
    "median_impressions": queue["impressions"].median(),
    "mean_impressions": queue["impressions"].mean(),
    "median_avg_position": queue["avg_position"].median(),
    "low_visibility_share": queue["low_visibility"].mean(),
    "weak_position_share": queue["weak_position"].mean(),
}])

display(monitoring_snapshot.round(3))

,snapshot_month,queue_rows,median_impressions,mean_impressions,median_avg_position,low_visibility_share,weak_position_share
0,2026-03,176738,173.0,1587.987,8.505,0.426,0.255


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exports for the paper

The notebook exports an aggregated ranked action queue and a monitoring summary to `work/outputs/`. These files support reproducibility and can be used to report the distribution of recommended review actions.

The exported queue contains hashed identifiers and performance aggregates, not public URLs or client names. The public paper should report aggregate findings and illustrative reason codes rather than exposing raw content-level records.

In [13]:
from pathlib import Path

output_dir = Path("/content/work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

queue.to_csv(output_dir / "ranked_action_queue.csv", index=False)
monitoring_snapshot.to_csv(output_dir / "monitoring_snapshot.csv", index=False)
reason_summary.to_csv(output_dir / "reason_code_summary.csv", index=False)

print("Files saved:")
for file in output_dir.glob("*.csv"):
    print(file.name, "-", file.stat().st_size, "bytes")

Files saved:
monitoring_snapshot.csv - 219 bytes
ranked_action_queue.csv - 26067698 bytes
reason_code_summary.csv - 117 bytes


In [14]:
from pathlib import Path

output_dir = Path("work/outputs")

print("Current folder:", Path.cwd())
print("CSV files found:")

for file in output_dir.glob("*.csv"):
    print(file)

Current folder: /content
CSV files found:
work/outputs/monitoring_snapshot.csv
work/outputs/ranked_action_queue.csv
work/outputs/reason_code_summary.csv


In [15]:
import pandas as pd

print("Dataframes currently available:\n")

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(f"{name}: {obj.shape}")
        print("Columns:", list(obj.columns))
        print()

Dataframes currently available:

monthly: (176738, 10)
Columns: ['client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'avg_position', 'low_visibility', 'weak_position', 'reason_code', 'priority_score', 'suggested_action']

queue: (176738, 11)
Columns: ['rank', 'client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'avg_position', 'low_visibility', 'weak_position', 'reason_code', 'priority_score', 'suggested_action']

reason_summary: (4, 2)
Columns: ['reason_code', 'items']

priority_summary: (3, 2)
Columns: ['priority_score', 'items']

monitoring_snapshot: (1, 7)
Columns: ['snapshot_month', 'queue_rows', 'median_impressions', 'mean_impressions', 'median_avg_position', 'low_visibility_share', 'weak_position_share']



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [16]:
import zipfile

with zipfile.ZipFile("/content/aggregate_outputs.zip", "w") as z:
    z.write("/content/work/outputs/monitoring_snapshot.csv",
            "monitoring_snapshot.csv")
    z.write("/content/work/outputs/reason_code_summary.csv",
            "reason_code_summary.csv")

from google.colab import files
files.download("/content/aggregate_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>